In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Grid parameters
n_qubits = 5
N = 2**n_qubits
L = 12.0
dx = L / N
x = np.linspace(-L/2, L/2, N, endpoint=False)

In [ ]:
# Momentum grid (DFT order)
dp = 2 * np.pi / L
p = np.fft.fftfreq(N) * N * dp
#print(p)

In [ ]:
# Initial wavepacket parameters
x0 = -3.0
sigma = 1.0
p0 = 0.0

# Gaussian wavepacket
psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * p0 * x)

# Normalize
psi = psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

In [ ]:
# Plot initial wavepacket
plt.figure(figsize=(10, 4))
plt.plot(x, np.abs(psi)**2)
plt.xlabel("x")
plt.ylabel("|psi(x)|^2")
plt.title("Initial wavepacket")
plt.grid(True)
plt.show()

In [ ]:
# Time parameters
dt = 0.05
n_steps = 200

# Kinetic propagator phases
kinetic_phases = np.exp(-1j * p**2 / 2 * dt)
# Half-potential propagator phases
potential_phases_half = np.exp(-1j * (x**2 / 2) * (dt / 2))

In [ ]:
# Time evolution with classical split-operator (FFT)
psi_t = psi.copy()
positions_classical = []
x_means_classical = []

for step in range(n_steps):
    # Apply potential propagator
    psi_t = psi_t * potential_phases_half
    
    # Transform to momentum space
    psi_p = np.fft.fft(psi_t)
    
    # Apply kinetic propagator
    psi_p = psi_p * kinetic_phases
    
    # Transform back to position space
    psi_t = np.fft.ifft(psi_p)

    # Apply potential propagator (again)
    psi_t = psi_t * potential_phases_half

    # Save probability density and x_mean
    positions_classical.append(np.abs(psi_t)**2)
    x_means_classical.append(np.sum(x * np.abs(psi_t)**2) * dx)

In [ ]:
# Quick visualization of classical evolution
t = np.arange(n_steps) * dt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: expectation value of position
axes[0].plot(t, x_means_classical)
axes[0].plot(t, x0 * np.cos(t), '--', label='x0*cos(t)')
axes[0].set_xlabel("t")
axes[0].set_ylabel("<x>(t)")
axes[0].set_title("Expectation value of position")
axes[0].legend()
axes[0].grid(True)

# Right: probability density as 2D image
positions_array = np.array(positions_classical)
axes[1].imshow(positions_array, aspect='auto', extent=[-L/2, L/2, n_steps*dt, 0])
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
axes[1].set_title("Probability density |psi(x,t)|^2")

plt.tight_layout()
plt.show()

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from qiskit.circuit.library import DiagonalGate
from qiskit.quantum_info import Statevector

In [ ]:
simulator = AerSimulator(method='statevector')

def qiskit_step(psi, potential_phases_half, kinetic_phases, n_qubits):
    
    qc = QuantumCircuit(n_qubits)
    qc.set_statevector(Statevector(psi))
    qc.append(DiagonalGate(list(potential_phases_half)), range(n_qubits))
    qc.append(QFTGate(n_qubits), range(n_qubits))
    qc.append(DiagonalGate(list(kinetic_phases)), range(n_qubits))
    qc.append(QFTGate(n_qubits).inverse(), range(n_qubits))
    qc.append(DiagonalGate(list(potential_phases_half)), range(n_qubits))
    qc.save_statevector()
    qc = transpile(qc, simulator)
    
    result = simulator.run(qc).result()
    psi_new = np.array(result.get_statevector())
    
    return psi_new

In [ ]:
# Verify single step: classical vs quantum
psi_norm = psi / np.sqrt(np.sum(np.abs(psi)**2))

# Classical single step
psi_fft = psi_norm * potential_phases_half
psi_fft = np.fft.ifft(np.fft.fft(psi_fft) * kinetic_phases)
psi_fft = psi_fft * potential_phases_half

# Quantum single step
psi_qiskit = qiskit_step(psi_norm, potential_phases_half, kinetic_phases, n_qubits)

print("Max difference:", np.max(np.abs(psi_qiskit - psi_fft)))

In [ ]:
# Time evolution with quantum split-operator (Qiskit)
psi_t = psi / np.sqrt(np.sum(np.abs(psi)**2))
positions_quantum = []
x_means_quantum = []

for step in range(n_steps):
    psi_t = qiskit_step(psi_t, potential_phases_half, kinetic_phases, n_qubits)
    positions_quantum.append(np.abs(psi_t)**2)
    x_means_quantum.append(np.sum(x * np.abs(psi_t)**2))

In [ ]:
# Visualization
t = np.arange(n_steps) * dt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: expectation value of position
axes[0].plot(t, x_means_quantum, label='Quantum')
axes[0].plot(t, x_means_classical, '--', label='Classical')
axes[0].plot(t, x0 * np.cos(t), ':', label='x0*cos(t)')
axes[0].set_xlabel("t")
axes[0].set_ylabel("<x>(t)")
axes[0].set_title("Expectation value of position")
axes[0].legend()
axes[0].grid(True)

# Right: probability density as 2D image
positions_array = np.array(positions_quantum)
axes[1].imshow(positions_array, aspect='auto', extent=[-L/2, L/2, n_steps*dt, 0])
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
axes[1].set_title("Probability density |psi(x,t)|^2 (Quantum)")

plt.tight_layout()
plt.show()